# New Section

In [1]:
# MLflow / DagsHub tracking setup
import os
import mlflow

os.environ["MLFLOW_TRACKING_USERNAME"] = "Roy7721"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "bc912b7d58bd2aca05abdc192e010414493a3886"  # replace before running

mlflow.set_tracking_uri("https://dagshub.com/Roy7721/yt_comment_analysis.mlflow")

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/ced16f5d8bce44b5a9c3e335b0f484b4', creation_time=1784571024655, effective_trace_archival_retention=None, experiment_id='9', last_update_time=1784571024655, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('dataset.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Step 1: Load and clean data
#df = pd.read_csv('/content/dataset.csv').dropna()
df = df.dropna(subset=['category'])

# Step 2: Split FIRST on raw text to avoid data leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Step 3: TF-IDF vectorizer - FIT ONLY on training data
ngram_range = (1, 2)  # Bigram
max_features = 10000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X_train = vectorizer.fit_transform(X_train_raw)  # Fit ONLY on training data
X_test = vectorizer.transform(X_test_raw)  # Transform test data (don't refit)

# Step 4: Apply ADASYN ONLY to training data to handle class imbalance
from imblearn.over_sampling import ADASYN
adasyn = ADASYN(random_state=42)
X_train_resampled, y_train_resampled = adasyn.fit_resample(X_train, y_train)

# Test data stays untouched for fair evaluation
X_train = X_train_resampled
y_train = y_train_resampled

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_ADASYN_TFIDF_Bigrams_NoLeakage")
        mlflow.set_tag("experiment_type", "algorithm_comparison")
        mlflow.set_tag("data_handling", "no_leakage_fixed")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)
        mlflow.log_param("vectorizer_type", "TfidfVectorizer")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)
        mlflow.log_param("imbalance_handling", "ADASYN_on_train_only")

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 5: Optuna objective function for Random Forest
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)

    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf,
                                   random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


# Step 6: Run Optuna for Random Forest, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_rf, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = RandomForestClassifier(n_estimators=best_params['n_estimators'],
                                        max_depth=best_params['max_depth'],
                                        min_samples_split=best_params['min_samples_split'],
                                        min_samples_leaf=best_params['min_samples_leaf'],
                                        random_state=42)

    # Log the best model with MLflow
    log_mlflow("RandomForest", best_model, X_train, X_test, y_train, y_test)
    
    print(f"Best Optuna parameters: {best_params}")
    print(f"Best trial accuracy: {study.best_value:.4f}")

# Run the experiment for Random Forest
run_optuna_experiment()

[I 2026-07-21 00:48:12,147] A new study created in memory with name: no-name-97836fa4-71b0-468a-a5a2-e9687aac94f7
[I 2026-07-21 00:48:16,800] Trial 0 finished with value: 0.5707077594436111 and parameters: {'n_estimators': 178, 'max_depth': 8, 'min_samples_split': 16, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.5707077594436111.
[I 2026-07-21 00:48:34,141] Trial 1 finished with value: 0.6413473339697259 and parameters: {'n_estimators': 267, 'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 6}. Best is trial 1 with value: 0.6413473339697259.
[I 2026-07-21 00:48:40,415] Trial 2 finished with value: 0.5749352243283785 and parameters: {'n_estimators': 206, 'max_depth': 8, 'min_samples_split': 12, 'min_samples_leaf': 16}. Best is trial 1 with value: 0.6413473339697259.
[I 2026-07-21 00:48:42,783] Trial 3 finished with value: 0.5092049638619938 and parameters: {'n_estimators': 115, 'max_depth': 4, 'min_samples_split': 18, 'min_samples_leaf': 11}. Best is trial 1 with va

🏃 View run RandomForest_ADASYN_TFIDF_Bigrams_NoLeakage at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9/runs/81534da03dee453d99bc01010ceb9de4
🧪 View experiment at: https://dagshub.com/Roy7721/yt_comment_analysis.mlflow/#/experiments/9
Best Optuna parameters: {'n_estimators': 52, 'max_depth': 20, 'min_samples_split': 6, 'min_samples_leaf': 1}
Best trial accuracy: 0.6568
